# `cost_eval_assets.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [2]:
# ---- 项目路径：向上找到项目根目录，把 src/ 加入 import 路径 ----
import sys
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "src" / "paths.py").exists():
    if _root == _root.parent:
        raise FileNotFoundError("找不到项目根目录（含 src/paths.py 的文件夹）")
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
from paths import DATA_RAW, RES_TRAINING, RES_EVALUATION

# cost_eval_assets.py
# 用「逐资产成本 c_i」评估 benchmark（Long-Only / Sgn / MACD-Baz）以及已训练的 MLP/LSTM，
# 分别在两个资产池上：
#   (A) 成本表第 1 页的全部标的   (B) 成本表第 2 页的高流动性标的
# 每日成本 = sum_i c_i * |w_i,t - w_i,t-1| / N_t   （论文式 35，把统一的 c 换成 c_i）
# 所有结果都缩放到 15% 组合波动率；成本倍数 0.5x / 1x / 2x 是成本表 ReadMe 建议的稳健性检验。
# 说明：每个资产池内部重新计算 N_t 并重新缩放波动率。MLP/LSTM 用的是「全池训练」得到的仓位，只是筛选标的，没有重训。

import os
import numpy as np
import pandas as pd

from neural_config import DATA_PATH, FEATURES, MACD_PAIRS

COST_FILE = DATA_RAW / "trading_cost_50_etfs_1.xlsx"
REF_FILE = RES_TRAINING / "lstm_oos_predictions.csv"          # benchmark 与它共用同一批 (Date, Ticker) 样本
NN_FILES = {"MLP": RES_TRAINING / "mlp_oos_predictions.csv", "LSTM": RES_TRAINING / "lstm_oos_predictions.csv"}
MULTIPLIERS = [0, 0.5, 1, 2]                   # 0 = 不扣成本

TRADING_DAYS = 252
TARGET_ANNUAL_VOL = 0.15
TARGET_DAILY_VOL = TARGET_ANNUAL_VOL / np.sqrt(TRADING_DAYS)

### `load_costs()`

In [3]:
def load_costs(path=COST_FILE):
    x = pd.ExcelFile(path)

    def pick(prefix):
        name = next(s for s in x.sheet_names if s.startswith(prefix))
        df = x.parse(name).iloc[:, :2]
        df.columns = ["Ticker", "c_bp"]
        s = df.dropna().set_index("Ticker")["c_bp"].astype(float)
        return s / 1e4                         # bp -> 小数

    return pick("Cost(50"), pick("Cost(高流动性")

### `phi()`

In [4]:
def phi(y):
    return y * np.exp(-(y ** 2) / 4) / 0.89

### `signal()`

In [5]:
def signal(df, rule):
    if rule == "long_only":
        return np.ones(len(df))
    if rule == "sgn_returns":
        return np.sign(df["norm_ret_1y"].to_numpy())
    if rule == "macd_baz":                     # 三个时间尺度取平均
        y = sum(df[f"macd_{s}_{l}"] for s, l in MACD_PAIRS) / len(MACD_PAIRS)
        return phi(y).to_numpy()
    raise ValueError(rule)

### `load_benchmark_base()`

In [6]:
def load_benchmark_base():
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["model_available"] == True].copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["vol_daily", "fwd_ret_1d"])
    df = df[df["vol_daily"] > 0]
    ref = pd.read_csv(REF_FILE, parse_dates=["Date"])[["Date", "Ticker"]]
    return df.merge(ref, on=["Date", "Ticker"], how="inner")

### `finish()`

In [7]:
def finish(p):
    p = p.sort_values(["Ticker", "Date"]).copy()
    p["n_t"] = p.groupby("Date")["Ticker"].transform("count")
    return p

### `rescale()`

In [8]:
def rescale(r, span=60):
    vol = r.ewm(span=span, adjust=False).std() * np.sqrt(TRADING_DAYS)
    return (r * (TARGET_ANNUAL_VOL / vol).shift(1)).dropna()

### `sharpe()`

In [9]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(TRADING_DAYS)

### `max_drawdown()`

In [10]:
def max_drawdown(r):
    w = (1 + r).cumprod()
    return float(-(w / w.cummax() - 1).min())

### `evaluate()`

In [11]:
def evaluate(p, weight, c_map):
    c_i = p["Ticker"].map(c_map).to_numpy(float)
    dw = weight.groupby(p["Ticker"]).diff().abs().fillna(0.0)
    gross = (weight * p["fwd_ret_1d"] / p["n_t"]).groupby(p["Date"]).sum()
    cost = (dw * c_i / p["n_t"]).groupby(p["Date"]).sum()
    turn = (dw / p["n_t"]).groupby(p["Date"]).sum()

    row = {"年化换手": turn.mean() * TRADING_DAYS,
           "成本占毛收益比例(1x)": cost.mean() / gross.mean() if gross.mean() > 0 else np.nan}
    for m in MULTIPLIERS:
        row[f"Sharpe {m}x"] = sharpe(rescale(gross - m * cost))
    r1 = rescale(gross - cost)
    row["收益(1x,缩放后)"] = r1.mean() * TRADING_DAYS
    row["MDD(1x)"] = max_drawdown(r1)
    return row

### `run_universe()`

In [12]:
def run_universe(name, tickers, c_map, base, nn_panels):
    rows = {}
    for label, rule in [("Long-Only", "long_only"), ("Sgn(Returns)", "sgn_returns"), ("MACD-Baz2015", "macd_baz")]:
        d = base[base["Ticker"].isin(tickers)].copy()
        d["signal"] = signal(d, rule)
        d = finish(d)
        w = d["signal"] * TARGET_DAILY_VOL / d["vol_daily"]
        rows[label] = evaluate(d, w, c_map)
    for label, panel in nn_panels.items():
        p = finish(panel[panel["Ticker"].isin(tickers)].copy())
        rows[f"{label}(全池训练,未重训)"] = evaluate(p, p["weight"], c_map)

    table = pd.DataFrame(rows).T
    print(f"\n===== {name}：{len(tickers)} 只 =====")
    print(table.round(3).to_string())
    table.to_csv(RES_EVALUATION / f"cost_assets_{name}.csv")
    return table

### `main()`

In [13]:
def main():
    c_all, c_liq = load_costs()
    base = load_benchmark_base()
    data_tickers = set(base["Ticker"].unique())

    miss = sorted(data_tickers - set(c_all.index))
    if miss:
        print(f"[提示] 以下标的在数据里但成本表第 1 页没有，已从两个资产池中排除：{miss}")

    nn_panels = {k: pd.read_csv(v, parse_dates=["Date"]) for k, v in NN_FILES.items() if os.path.exists(v)}

    all_t = sorted(data_tickers & set(c_all.index))
    liq_t = sorted(data_tickers & set(c_liq.index))
    run_universe("全部", all_t, c_all, base, nn_panels)
    run_universe("高流动性", liq_t, c_liq, base, nn_panels)

    print("\n说明：成本表只含买卖价差的一半，不含佣金、融券费、冲击成本；当前价差用于全样本会低估早期成本。")

### Entry point

In [14]:
if __name__ == "__main__":
    main()

[提示] 以下标的在数据里但成本表第 1 页没有，已从两个资产池中排除：['FXS']

===== 全部：50 只 =====
                  年化换手  成本占毛收益比例(1x)  Sharpe 0x  Sharpe 0.5x  Sharpe 1x  Sharpe 2x  收益(1x,缩放后)  MDD(1x)
Long-Only        4.520         0.079      0.501        0.478      0.455      0.409       0.071    0.378
Sgn(Returns)    16.365         0.547      0.431        0.306      0.181     -0.069       0.028    0.454
MACD-Baz2015     8.375         0.366      0.380        0.303      0.226      0.072       0.035    0.438
MLP(全池训练,未重训)   79.895         3.045      0.637       -0.398     -1.424     -3.416      -0.228    0.994
LSTM(全池训练,未重训)  36.183         1.912      0.654       -0.161     -0.971     -2.558      -0.152    0.975

===== 高流动性：21 只 =====
                  年化换手  成本占毛收益比例(1x)  Sharpe 0x  Sharpe 0.5x  Sharpe 1x  Sharpe 2x  收益(1x,缩放后)  MDD(1x)
Long-Only        4.053         0.009      0.820        0.816      0.812      0.805       0.126    0.302
Sgn(Returns)    12.323         0.058      0.541        0.523      0.505      0.4